# Feature engineering test: raw sensors vs engineered features (issue #14)

Question: do extra features help boosting models on this data?

Follows the shared evaluation protocol (`src/common.py`, #8): development data = train + val, repeated stratified 5-fold CV x 3, seed 42, test set untouched. Every run is logged to `results/model_results.csv` with the feature set in the `features` column.

Models: `HistGradientBoosting` (defaults) and `CatBoost` with the parameters tuned on the raw sensors in #13 (CatBoost tuned was the best single model so far; see `boosting_catboost.ipynb`). Note the CatBoost parameters were tuned on raw features, which if anything favours the raw set.

Feature sets:
- `raw`: the 6 sensors
- `all`: raw + the 16 engineered columns shipped in the Hugging Face `train.csv`
- `raw+own`: raw + 4 features of our own
- `all, corr-dropped`: `all` minus one column of every pair with |correlation| > 0.95
- `all, top-k`: `all` with univariate selection (mutual information) done **inside** each CV training fold

In [1]:
import sys, json, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np, pandas as pd
from catboost import CatBoostClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.model_selection import cross_validate
from sklearn.pipeline import make_pipeline

from src.common import load_dev, evaluate, get_cv, RANDOM_STATE, RAW_FEATURES, ROOT
warnings.filterwarnings('ignore')

X_raw, y = load_dev('raw')
X_all, _ = load_dev('all')
cat_params = json.loads((ROOT / 'results' / 'best_params_catboost.json').read_text())
print(X_raw.shape, X_all.shape); print(cat_params)

(16604, 6) (16604, 22)
{'border_count': 95, 'depth': 2, 'iterations': 230, 'l2_leaf_reg': 13.146443027030747, 'learning_rate': 0.0767539109585377}


## 1. Leakage check

Every engineered column must be computed **row by row from the six sensors** (no target, no other rows, no global statistics). We rebuild each one from the sensors and compare. The tiny remaining errors (below 0.05%) come from rounding in the stored columns where a ratio divides by a pressure near 0.003.

In [2]:
r = X_all
rpm, oilp, fuelp, coolp, oilt, coolt = (r[c] for c in RAW_FEATURES)
rebuilt = {
 'RPM_x_OilPressure': rpm*oilp, 'RPM_x_FuelPressure': rpm*fuelp, 'RPM_x_CoolantPressure': rpm*coolp,
 'OilTemp_x_OilPressure': oilt*oilp, 'CoolantTemp_x_CoolantPressure': coolt*coolp,
 'RPM_squared': rpm**2, 'OilPressure_squared': oilp**2, 'TempDiff': oilt-coolt,
 'OilFuelPressureRatio': oilp/fuelp, 'CoolantOilPressureRatio': coolp/oilp,
 'OilHealthIndex': oilp/oilt, 'CoolantStress': coolt/coolp,
 'OilTempPerRPM': oilt/rpm, 'CoolantTempPerRPM': coolt/rpm,
 'PressureSum': oilp+fuelp+coolp, 'TempSum': oilt+coolt}
chk = pd.DataFrame({c: {'max relative error': float(np.max(np.abs(r[c]-v)/np.abs(v))), 'row-wise formula OK': bool(np.allclose(r[c], v, rtol=1e-3, atol=1e-8))}
                    for c, v in rebuilt.items()}).T
print(chk['row-wise formula OK'].value_counts().to_dict(), '(of 16)')
chk

{True: 16} (of 16)


,max relative error,row-wise formula OK
RPM_x_OilPressure,0.0,True
RPM_x_FuelPressure,0.0,True
RPM_x_CoolantPressure,0.0,True
OilTemp_x_OilPressure,0.0,True
CoolantTemp_x_CoolantPressure,0.0,True
RPM_squared,0.0,True
OilPressure_squared,0.0,True
TempDiff,0.0,True
OilFuelPressureRatio,0.000314,True
CoolantOilPressureRatio,0.000295,True


## 2. Our own features

After the EDA (RPM and the per-RPM temperatures carry most of the weak signal) we add four more row-wise features:

In [3]:
own = pd.DataFrame({
    'OilTemp_over_CoolantTemp': oilt/coolt,
    'FuelPressure_per_RPM': fuelp/rpm,
    'FuelOverCoolantPressure': fuelp/coolp,
    'LogRPM': np.log(rpm)}, index=X_raw.index)
X_own = pd.concat([X_raw, own], axis=1)
X_own.shape

(16604, 10)

## 3. Correlated columns

Drop one column from every pair of the `all` set with |correlation| > 0.95 (done on the dev data's features only, never the target).

In [4]:
corr = X_all.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
drop = [c for c in upper.columns if (upper[c] > 0.95).any()]
X_dropped = X_all.drop(columns=drop)
print('dropped', len(drop), ':', drop); print('kept', X_dropped.shape[1])

dropped 6 : ['OilTemp_x_OilPressure', 'CoolantTemp_x_CoolantPressure', 'RPM_squared', 'OilPressure_squared', 'OilHealthIndex', 'CoolantTempPerRPM']
kept 16


## 4. Run everything through the shared CV (logged)

In [5]:
def cat():  return CatBoostClassifier(**cat_params, random_seed=RANDOM_STATE, thread_count=1, verbose=0, allow_writing_files=False)
def hgb():  return HistGradientBoostingClassifier(random_state=RANDOM_STATE)
def topk(k, make): return make_pipeline(SelectKBest(mutual_info_classif, k=k), make())

runs = [('raw', X_raw, None), ('all', X_all, None), ('raw+own', X_own, None), ('all, corr-dropped', X_dropped, None),
        ('all, top-8', X_all, 8), ('all, top-12', X_all, 12)]
rows = []
for label, X, k in runs:
    for mname, make in [('HistGradientBoosting', hgb), ('CatBoost (params from #13)', cat)]:
        model = topk(k, make) if k else make()
        note = '#14 feature test' + ('; SelectKBest(mutual_info) inside CV' if k else '')
        rows.append(evaluate(model, X, y, name=mname, params={'defaults': True} if mname.startswith('Hist') else cat_params,
                             features=label, notes=note, n_jobs=-1))
tbl = pd.DataFrame(rows)[['model', 'features', 'cv_auc_mean', 'cv_auc_std', 'cv_pr_auc_mean', 'cv_f1_mean', 'cv_recall_mean', 'fit_time_mean_s']]
tbl.sort_values(['model', 'cv_auc_mean'], ascending=[True, False])

,model,features,cv_auc_mean,cv_auc_std,cv_pr_auc_mean,cv_f1_mean,cv_recall_mean,fit_time_mean_s
1,CatBoost (params from #13),raw,0.7036,0.0083,0.8008,0.7634,0.8429,0.391
3,CatBoost (params from #13),all,0.7028,0.0087,0.8000,0.7621,0.8415,0.457
5,CatBoost (params from #13),raw+own,0.7025,0.0091,0.8000,0.7604,0.8333,0.342
7,CatBoost (params from #13),"all, corr-dropped",0.7022,0.0082,0.7996,0.7616,0.8420,0.371
11,CatBoost (params from #13),"all, top-12",0.7001,0.0087,0.7974,0.7623,0.8447,0.774
9,CatBoost (params from #13),"all, top-8",0.6949,0.0104,0.7934,0.7622,0.8528,0.742
0,HistGradientBoosting,raw,0.6957,0.0100,0.7950,0.7585,0.8367,0.072
4,HistGradientBoosting,raw+own,0.6952,0.0092,0.7948,0.7588,0.8350,0.070
2,HistGradientBoosting,all,0.6945,0.0092,0.7941,0.7577,0.8348,0.137
6,HistGradientBoosting,"all, corr-dropped",0.6943,0.0101,0.7941,0.7578,0.8362,0.090


## 5. Is the difference real? Paired fold-by-fold comparison

Same 15 CV splits for every feature set, so we can compare the AUC of the same fold between `raw` and the other sets. A feature set only helps if its mean paired difference is clearly larger than its standard error.

In [6]:
def fold_auc(model, X):
    return cross_validate(model, X, y, cv=get_cv(), scoring='roc_auc', n_jobs=-1)['test_score']

base = {'HistGradientBoosting': fold_auc(hgb(), X_raw), 'CatBoost': fold_auc(cat(), X_raw)}
sets = {'all': X_all, 'raw+own': X_own, 'all, corr-dropped': X_dropped}
out = []
for label, X in sets.items():
    for mname, make in [('HistGradientBoosting', hgb), ('CatBoost', cat)]:
        d = fold_auc(make(), X) - base[mname]
        out.append({'model': mname, 'feature set vs raw': label, 'mean AUC diff': round(d.mean(), 4),
                    'std err': round(d.std(ddof=1)/np.sqrt(len(d)), 4), 'folds better': f'{(d>0).sum()}/{len(d)}'})
print('(folds are 3 repeats of 5-fold, so folds are not fully independent; the std err is optimistic)')
pd.DataFrame(out)

(folds are 3 repeats of 5-fold, so folds are not fully independent; the std err is optimistic)


,model,feature set vs raw,mean AUC diff,std err,folds better
0,HistGradientBoosting,all,-0.0011,0.0007,6/15
1,CatBoost,all,-0.0008,0.0004,5/15
2,HistGradientBoosting,raw+own,-0.0005,0.0008,7/15
3,CatBoost,raw+own,-0.0012,0.0004,3/15
4,HistGradientBoosting,"all, corr-dropped",-0.0013,0.0009,4/15
5,CatBoost,"all, corr-dropped",-0.0014,0.0003,2/15


## Conclusion

**Use the 6 raw sensors for the final comparison.** The 16 shipped engineered columns are valid (each is a row-wise formula of the sensors, no leakage) but they add nothing a boosting model can use. With CatBoost, `all` scores 0.7028 AUC vs 0.7036 for `raw`; with HistGradientBoosting, 0.6945 vs 0.6957. Our own four features, dropping correlated columns and univariate selection do not beat raw either. The paired fold-by-fold differences are all slightly negative (about -0.001 AUC, never positive) and far smaller than the spread between folds (std about 0.009). Keeping only 8 or 12 features is worse, and extra columns make fitting slower for HistGB (0.07 s on raw vs 0.14 s on all). Trees can already form products and ratios themselves; the weak signal in this data (AUC about 0.70) is what limits every model, not the feature set.

Caveat: CatBoost's parameters were tuned on the raw sensors in #13, so that comparison is slightly tilted toward `raw`; the untuned HistGradientBoosting defaults point the same way.